# 🚀 Bengali GPT 50M — Latest GGUF Generator & Quantizer
### 🎯 এই Colab Notebook দিয়ে আপনি আপনার সর্বশেষ ট্রেইনড `.pt` চেকপয়েন্ট থেকে অতি দ্রুত অ্যান্ড্রয়েড-রেডি `.gguf` তৈরি ও ডাউনলোড করতে পারবেন।

- **আউটপুট ১:** `bengali_gpt_50m_q8_0.gguf` (~86 MB — সর্বোচ্চ নির্ভুলতা ও মান)
- **আউটপুট ২:** `bengali_gpt_50m_q4_k_m.gguf` (~32 MB — অতি দ্রুত ও কম RAM ব্যবহার)
- **উপযোগিতা:** itel A60 / A662L (32-bit armeabi-v7a) এবং যেকোনো অ্যান্ড্রয়েড ফোনে সরাসরি অফলাইনে সেকেন্ডের মধ্যে রেসপন্স দেয়!

In [ ]:
# Step 1: প্রয়োজনীয় লাইব্রেরি ইনস্টল
!pip install -q gguf torch safetensors
print('✓ লাইব্রেরি ইনস্টলেশন সফল!')

In [ ]:
# Step 2: Google Drive মাউন্ট করুন (যদি আপনার চেকপয়েন্ট ড্রাইভে থাকে)
import os
from google.colab import drive
try:
    drive.mount('/content/drive')
    print('✓ Google Drive মাউন্ট সফল!')
except Exception as e:
    print('Drive মাউন্ট স্কিপ করা হয়েছে, আপনি সরাসরি Colab-এ ফাইল আপলোড করতে পারবেন।')

In [ ]:
# Step 3: কোডবেস ডাউনলোড অথবা ক্লোন
%cd /content
!rm -rf ss_100m
!git clone https://github.com/kajshikhi49-afk/ss_100m.git
%cd /content/ss_100m/ss_50million
print('✓ রিপোজিটরি প্রস্তুত!')

In [ ]:
# Step 4: চেকপয়েন্ট (.pt) পাথ নির্বাচন
import os

# ড্রাইভে ফাইল থাকলে পাথটি দিন, অথবা Colab-এর Files ট্যাবে আপলোড করুন:
CHECKPOINT_PATH = '/content/drive/MyDrive/bengali_gpt_50m_checkpoints/checkpoint_stage_2_final.pt'

candidates = [
    '/content/drive/MyDrive/bengali_gpt_50m_checkpoints/checkpoint_stage_2_final.pt',
    '/content/drive/MyDrive/checkpoint_stage_2_final.pt',
    '/content/checkpoint_stage_2_final.pt',
    '/content/checkpoint.pt',
    '/content/drive/MyDrive/bengali_gpt_50m_checkpoints/checkpoint_stage_1.pt'
]
for c in candidates:
    if os.path.exists(c):
        CHECKPOINT_PATH = c
        break

print(f'🎯 নির্বাচিত চেকপয়েন্ট: {CHECKPOINT_PATH}')
if not os.path.exists(CHECKPOINT_PATH):
    print('⚠️ চেকপয়েন্ট ফাইলটি এখনও খুঁজে পাওয়া যায়নি!')
    print('👉 আপনি Colab-এর বাম পাশের ফাইল আইকন (Files) এ ক্লিক করে সরাসরি আপনার .pt ফাইলটি /content এ ড্র্যাগ করে আপলোড করতে পারেন।')

In [ ]:
# Step 5: ফিক্সড GGUF রূপান্তর (SuperBPE + RoPE Interleaving সহ)
import sys, os, shutil
sys.path.append('/content/ss_100m/ss_50million')
from scripts.export_model_fixed import export_to_gguf_fixed

raw_f16 = '/content/bengali_gpt_50m_f16.gguf'
export_to_gguf_fixed(CHECKPOINT_PATH, raw_f16, 'tokenizer.json')
print(f'✓ F16 GGUF তৈরি সম্পন্ন! সাইজ: {os.path.getsize(raw_f16)/(1024*1024):.1f} MB')

In [ ]:
# Step 6: llama.cpp দিয়ে Q8_0 এবং Q4_K_M কোয়ান্টাইজেশন তৈরি
print('⚡ llama-quantize কম্পাইল হচ্ছে...')
if not os.path.exists('/content/llama.cpp'):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
!cd /content/llama.cpp && cmake -B build && cmake --build build --config Release -t llama-quantize llama-cli -j4

quant_bin = '/content/llama.cpp/build/bin/llama-quantize'
if not os.path.exists(quant_bin):
    quant_bin = '/content/llama.cpp/llama-quantize'

q8_file = '/content/bengali_gpt_50m_q8_0.gguf'
q4_file = '/content/bengali_gpt_50m_q4_k_m.gguf'

# Q8_0 তৈরি (রেকমেন্ডেড)
!{quant_bin} {raw_f16} {q8_file} q8_0

# Q4_K_M তৈরি (লাইটওয়েট)
!{quant_bin} {raw_f16} {q4_file} q4_k_m

print('\n🎉 উভয় কোয়ান্টাইজড GGUF প্রস্তুত:')
print(f'1. {q8_file} ({os.path.getsize(q8_file)/(1024*1024):.1f} MB)')
print(f'2. {q4_file} ({os.path.getsize(q4_file)/(1024*1024):.1f} MB)')

In [ ]:
# Step 7: Colab-এই লাইভ টেস্ট করুন
cli_bin = '/content/llama.cpp/build/bin/llama-cli'
if not os.path.exists(cli_bin):
    cli_bin = '/content/llama.cpp/llama-cli'

print('=' * 65)
print('🤖 লাইভ টেস্ট আউটপুট:')
print('=' * 65)
!{cli_bin} -m {q8_file} -p "প্রশ্ন: বাংলাদেশের রাজধানী কী? উত্তর:" -n 50 --temp 0.5

In [ ]:
# Step 8: আপনার কম্পিউটারে GGUF ডাউনলোড করুন
from google.colab import files
print('⬇️ আপনার কম্পিউটারে bengali_gpt_50m_q8_0.gguf ডাউনলোড শুরু হচ্ছে...')
files.download(q8_file)